# Qwen-based phonetic embeddings (same input/output contract)

This notebook keeps the **same external contract** as the original BiLSTM notebook:

- **Input file:** `converted_phrases.csv`
- **Required columns:** `word`, `ipa`
- **Main output artifact:** `embedding_matrix.npy`
- **Same retrieval interface:** `retrieve_similar_words(query_word, top_k=10)`

The encoder is upgraded from a small BiLSTM to **`Qwen/Qwen3-Embedding-0.6B`**, while preserving the original data flow.


In [ ]:
# If needed, install dependencies in this notebook kernel
# !pip install -U sentence-transformers transformers accelerate faiss-cpu pandas numpy torch panphon tqdm


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import faiss
import panphon

from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

ft = panphon.FeatureTable()


In [ ]:
def ipa_to_feature_vector(ipa_string):
    segments = ft.ipa_segs(ipa_string)
    mapping = {'+': 1, '-': -1, '0': 0}

    vectors = [
        [mapping[feat] for feat in ft.segment_to_vector(seg)]
        for seg in segments
    ]
    return np.array(vectors, dtype=float)

ipa = 'ynivɛʁsalizəʁɔ̃'  # bonjour
vec = ipa_to_feature_vector(ipa)

print(f"IPA: {ipa}")
print(f"Feature vector: {vec}")


In [ ]:
# Original phonetic similarity helpers retained from the first notebook.
# These are useful for evaluation / sanity checks and keep the notebook behavior close to the original.

def phoneme_feature_set(segment):
    features = ft.names
    vector = ft.segment_to_vector(segment)
    return {features[i] for i, val in enumerate(vector) if val == '+'}

def phoneme_feature_set_extended(segment):
    if segment == 'BEG':
        return {'beg'}
    elif segment == 'END':
        return {'end'}
    else:
        return phoneme_feature_set(segment)

def bigram_feature_set(p1, p2):
    return phoneme_feature_set_extended(p1).union(phoneme_feature_set_extended(p2))

def bigram_similarity(bg1, bg2):
    set1 = bigram_feature_set(*bg1)
    set2 = bigram_feature_set(*bg2)
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    return intersection / union if union else 0.0

def ipa_to_bigrams(ipa):
    phonemes = ft.ipa_segs(ipa)
    phonemes = ['BEG'] + phonemes + ['END']
    return list(zip(phonemes[:-1], phonemes[1:]))

def word_similarity_bigrams(ipa1, ipa2):
    seq1 = ipa_to_bigrams(ipa1)
    seq2 = ipa_to_bigrams(ipa2)
    n, m = len(seq1), len(seq2)
    dp = np.zeros((n + 1, m + 1))

    for i in range(1, n + 1):
        dp[i, 0] = dp[i - 1, 0]
    for j in range(1, m + 1):
        dp[0, j] = dp[0, j - 1]

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            sim = bigram_similarity(seq1[i - 1], seq2[j - 1])
            dp[i, j] = max(
                dp[i - 1, j - 1] + sim,  # match/substitution
                dp[i - 1, j],            # deletion
                dp[i, j - 1]             # insertion
            )

    max_len = max(n, m)
    return dp[n, m] / max_len if max_len else 0.0

ipa1 = 'tɥa'
ipa2 = 'tɥa'
similarity_score = word_similarity_bigrams(ipa1, ipa2)
print(f"similarity: {similarity_score:.3f}")
print(ipa_to_bigrams(ipa1))
print(ipa_to_bigrams(ipa2))


In [ ]:
# SAME INPUT AS THE ORIGINAL NOTEBOOK
df = pd.read_csv("converted_phrases.csv")

required_cols = {"word", "ipa"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}. Expected the same schema as the original notebook.")

df = df.dropna(subset=["word", "ipa"]).reset_index(drop=True)

words = df["word"].astype(str).tolist()
ipas = df["ipa"].astype(str).tolist()

word_to_idx = {word: idx for idx, word in enumerate(words)}

print("rows:", len(df))
print("columns:", list(df.columns))
df.head()


## Qwen encoder

This notebook uses `Qwen/Qwen3-Embedding-0.6B` because it is dramatically stronger than the original BiLSTM while remaining much more practical than the 4B/8B variants for local or notebook use. The Qwen3 embedding family provides dedicated embedding models in 0.6B, 4B, and 8B sizes, with the 8B model ranking first on the multilingual MTEB leaderboard when released. citeturn237667search0turn237667search4

We keep the **same notebook inputs and outputs**; only the embedding backend changes.


In [ ]:
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"

# A tiny formatting helper: same IPA input, slightly richer text for the embedding model.
# This does NOT change the notebook input schema.
def ipa_to_qwen_text(ipa: str) -> str:
    segs = ft.ipa_segs(ipa)
    spaced = " ".join(segs) if segs else ipa
    return f"Represent this IPA pronunciation for phonetic similarity search: {ipa} | segmented: {spaced}"

model = SentenceTransformer(MODEL_NAME, device=device)
print("Loaded:", MODEL_NAME)


In [ ]:
# Sanity-check the transformed text that gets embedded
sample_ipa = ipas[0]
print("raw ipa:", sample_ipa)
print("model text:", ipa_to_qwen_text(sample_ipa))


In [ ]:
def encode_all_words_batched(ipas, batch_size=64, normalize_embeddings=True):
    texts = [ipa_to_qwen_text(ipa) for ipa in ipas]
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=normalize_embeddings,
    )
    return embeddings.astype("float32")

embedding_matrix = encode_all_words_batched(ipas)
print("embedding_matrix.shape =", embedding_matrix.shape)


In [ ]:
# SAME FAISS RETRIEVAL FLOW AS THE ORIGINAL NOTEBOOK
faiss.normalize_L2(embedding_matrix)
index = faiss.IndexFlatIP(embedding_matrix.shape[1])
index.add(embedding_matrix)


In [ ]:
word_to_idx = {word: i for i, word in enumerate(words)}

def retrieve_similar_words(query_word, top_k=10):
    if query_word not in word_to_idx:
        raise ValueError(f"'{query_word}' not found in vocabulary.")

    query_idx = word_to_idx[query_word]
    query_emb = embedding_matrix[query_idx].reshape(1, -1).astype("float32")

    faiss.normalize_L2(query_emb)
    similarities, indices = index.search(query_emb, top_k + 1)

    results = []
    for j, i in enumerate(indices[0]):
        if i != query_idx:
            results.append((words[i], float(similarities[0][j])))

    return results[:top_k]


In [ ]:
results = retrieve_similar_words("bonjour", top_k=10)
for word, score in results:
    print(f"  {word} (score: {score:.3f})")


## Optional quick sanity check against the original phonetic similarity signal

This cell is optional. It compares a few nearest neighbors from the Qwen embedding space against the original PanPhon-style bigram similarity used in the BiLSTM notebook.


In [ ]:
query_word = "bonjour"
results = retrieve_similar_words(query_word, top_k=5)

query_ipa = ipas[word_to_idx[query_word]]
print("Query:", query_word, "| IPA:", query_ipa)
print()

for word, score in results:
    cand_ipa = ipas[word_to_idx[word]]
    phon_score = word_similarity_bigrams(query_ipa, cand_ipa)
    print(f"{word:20s} emb_score={score:.3f}  phon_score={phon_score:.3f}  ipa={cand_ipa}")


In [ ]:
# SAME OUTPUT ARTIFACT NAME AS THE ORIGINAL NOTEBOOK
np.save("embedding_matrix.npy", embedding_matrix)

# Keep a model artifact on disk too. Since Qwen weights are large and come from Hugging Face,
# we save metadata/config here rather than serializing the full base model weights.
encoder_artifact = {
    "encoder_type": "qwen_embedding",
    "model_name": MODEL_NAME,
    "input_csv": "converted_phrases.csv",
    "required_columns": ["word", "ipa"],
    "embedding_shape": list(embedding_matrix.shape),
}
torch.save(encoder_artifact, "bilstm_encoder.pth")

with open("qwen_encoder_config.json", "w") as f:
    json.dump(encoder_artifact, f, indent=2)

print("Saved embedding_matrix.npy")
print("Saved bilstm_encoder.pth (metadata stub for compatibility)")
print("Saved qwen_encoder_config.json")
